# Barrilito — spike de portada (no es Hito 3)

Notebook para **fijar** interpolación, copy, gráficos y tokens visuales antes de Next.js + Tremor.

- Lee **solo** CSVs en `data/` (snapshots de `marts_cap4_dev`). Cero `raw_*`. Cero secretos.
- **No** es el dashboard público. El ADR 0001 sigue: Hito 3 = Next + Tremor.
- Completaciones: empty state. Adjunto IV no está en el job Meltano.

Marca UI: **Barrilito**. Repo: `vaca-muerta-pulse`.


## 1. Tokens (contrato visual para Front)

Paleta “crudo / asfalto / llama”. Un acento, no un arcoíris. Tipografía del spike: DejaVu (portable). En Next: **IBM Plex Sans** para UI, **Source Serif 4** para el número del contador.

| Token | Hex | Uso |
| --- | --- | --- |
| `bg` | `#141210` | fondo |
| `surface` | `#1E1B18` | cards |
| `text` | `#F3EDE3` | títulos y números |
| `muted` | `#A89F91` | disclaimer, ejes, unidades |
| `accent` | `#E0A04A` | petróleo / headline |
| `accent2` | `#C45C26` | hover / segundo rango |
| `line` | `#6F9B8F` | gas / serie secundaria |
| `grid` | `#2A2724` | grilla |
| `empty` | `#7A6A5A` | empty state |

**Reglas**

- Petróleo de headline siempre en **bbl** (ya convertido en el mart).
- Otras vistas: **m³** (petróleo/agua) y **miles de m³** (gas). No mezclar en el mismo eje.
- Disclaimer MUST al lado del contador, no en el footer: *simulación a partir de datos mensuales oficiales*.
- Cero copy de sensor, SCADA, “en este segundo”, “tiempo real”.
- Números con unidad visible. Separador de miles. 0 decimales en el contador live; 1 decimal en bbl/día.


In [ ]:
from pathlib import Path
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

DATA = Path("data")
ART = ZoneInfo("America/Argentina/Buenos_Aires")
AS_OF = datetime(2026, 9, 11, 19, 0, tzinfo=ZoneInfo("UTC")).astimezone(ART)

BARRILITO = {
    "bg": "#141210",
    "surface": "#1E1B18",
    "text": "#F3EDE3",
    "muted": "#A89F91",
    "accent": "#E0A04A",
    "accent2": "#C45C26",
    "line": "#6F9B8F",
    "grid": "#2A2724",
    "empty": "#7A6A5A",
}

plt.rcParams.update({
    "figure.facecolor": BARRILITO["bg"],
    "axes.facecolor": BARRILITO["surface"],
    "axes.edgecolor": BARRILITO["grid"],
    "axes.labelcolor": BARRILITO["muted"],
    "axes.titlecolor": BARRILITO["text"],
    "text.color": BARRILITO["text"],
    "xtick.color": BARRILITO["muted"],
    "ytick.color": BARRILITO["muted"],
    "grid.color": BARRILITO["grid"],
    "grid.linewidth": 0.6,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "legend.facecolor": BARRILITO["surface"],
    "legend.edgecolor": BARRILITO["grid"],
    "legend.labelcolor": BARRILITO["text"],
})

def fmt_miles(x, _pos=None):
    return f"{x:,.0f}".replace(",", " ")

def thousands_axis():
    return FuncFormatter(lambda x, p: fmt_miles(x))

headline = pd.read_csv(DATA / "headline.csv", parse_dates=["periodo"])
monthly = pd.read_csv(DATA / "monthly_pulse.csv", parse_dates=["periodo"])
companies = pd.read_csv(DATA / "company_latest.csv")
areas = pd.read_csv(DATA / "area_latest.csv")
top5 = pd.read_csv(DATA / "company_top5_month.csv", parse_dates=["periodo"])

row = headline.iloc[0]
print("as_of ART", AS_OF.isoformat())
print("periodo", row.periodo.date(), "rate_bbl_dia", round(row.rate_bbl_dia, 1), row.rate_method)
print("disclaimer:", row.disclaimer)
print("n companies", len(companies), "n areas", len(areas), "months", len(monthly))


## 2. Interpolación (decisión de producto)

El mes Cap. IV del mart es **diciembre 2025** y ya cerró. Un odómetro desde `periodo` hasta hoy (sep-2026) inflaría ~280 días × 260 bbl/día ≈ 73k barriles *inventados*. Eso miente.

**Elegimos el reloj diario (P0):**

```text
as_of en America/Argentina/Buenos_Aires
midnight = 00:00 del día civil de as_of
seconds = min(as_of - midnight, 86400)
barriles_hoy = rate_bbl_dia × seconds / 86400
```

- El número **reinicia a medianoche**.
- El ritmo es el del último mes oficial (`tef_weighted` acá).
- Copy junto al número: ritmo + mes fuente + disclaimer MUST.
- KPI aparte: volumen DDJJ del mes en **m³** (`prod_pet_m3`). El browser **no** multiplica `6.28981077`.

**No elegir para v1**

| Modelo | Por qué no |
| --- | --- |
| Odómetro desde `periodo` | El mes ya cerró; el total se va al infinito |
| `rate_bbl_dia × days_in_month` como “producción del mes en bbl” | Distinto de `prod_pet_m3 × factor` (tef vs calendario). El mart no expone ese total en bbl |
| Tick por pozo | No hay telemetría |


In [ ]:
rate = float(row.rate_bbl_dia)
midnight = datetime.combine(AS_OF.date(), time.min, tzinfo=ART)
elapsed = min((AS_OF - midnight).total_seconds(), 86400.0)
barrels_today = rate * elapsed / 86400.0
pace_per_sec = rate / 86400.0

print(f"ritmo {rate:.1f} bbl/día  ({pace_per_sec:.6f} bbl/s)")
print(f"desde {midnight.isoformat()} → {AS_OF.isoformat()}")
print(f"barriles simulados hoy: {barrels_today:,.1f}")
print("MUST:", row.disclaimer)
print("no es telemetría; is_simulation =", bool(row.is_simulation))


## 3. Portada — contador + KPIs

Layout que Tremor debe copiar: **un número enorme**, disclaimer a la derecha o debajo inmediato, tres KPIs (ritmo, volumen DDJJ m³, pozos con petróleo). Serie mensual abajo.


In [ ]:
fig = plt.figure(figsize=(11, 7), constrained_layout=True)
gs = fig.add_gridspec(3, 3, height_ratios=[1.35, 0.55, 1.6])

hero = fig.add_subplot(gs[0, :])
hero.set_axis_off()
hero.set_facecolor(BARRILITO["bg"])
hero.text(0.0, 0.92, "BARRILITO", fontsize=13, color=BARRILITO["accent"], fontweight="bold")
hero.text(0.0, 0.78, "Vaca Muerta · no convencional", fontsize=11, color=BARRILITO["muted"])
hero.text(0.0, 0.42, f"{barrels_today:,.0f}".replace(",", " "), fontsize=48, color=BARRILITO["text"], fontweight="bold", va="center")
hero.text(0.62, 0.42, "bbl hoy", fontsize=16, color=BARRILITO["accent"], va="center")
hero.text(
    0.0, 0.08,
    f"Ritmo {rate:.1f} bbl/día · {row.rate_method} · Capítulo IV {int(row.anio)}-{int(row.mes):02d}\n"
    f"{row.disclaimer}",
    fontsize=11, color=BARRILITO["muted"], va="bottom",
)

kpi_specs = [
    (f"{rate:.1f}", "bbl/día", "tasa del mart (no recalcular)"),
    (fmt_miles(row.prod_pet_m3), "m³ petróleo", "DDJJ dic-2025 (prod_pet_m3)"),
    (fmt_miles(row.well_month_row_count), "well-months", "recorte Pulse en el mes"),
]
for i, (value, unit, caption) in enumerate(kpi_specs):
    ax = fig.add_subplot(gs[1, i])
    ax.set_axis_off()
    ax.set_facecolor(BARRILITO["surface"])
    ax.text(0.06, 0.70, value, fontsize=16, color=BARRILITO["text"], fontweight="bold", transform=ax.transAxes)
    ax.text(0.06, 0.42, unit, fontsize=10, color=BARRILITO["accent"], transform=ax.transAxes)
    ax.text(0.06, 0.16, caption, fontsize=8, color=BARRILITO["muted"], transform=ax.transAxes)

ax = fig.add_subplot(gs[2, :])
ax.plot(monthly["periodo"], monthly["prod_pet_m3"] / 1e6, color=BARRILITO["accent"], linewidth=2.4, marker="o", markersize=5)
ax.fill_between(monthly["periodo"], monthly["prod_pet_m3"] / 1e6, color=BARRILITO["accent"], alpha=0.18)
ax.set_title("Petróleo mensual · recorte Pulse (millones de m³)")
ax.set_ylabel("10⁶ m³")
ax.grid(True, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(bottom=0)

fig.suptitle("Portada P0 — el disclaimer viaja pegado al número", color=BARRILITO["muted"], fontsize=10, x=0.01, ha="left")
plt.show()


## 4. Ranking de empresas (último mes Cap. IV)

Grano `idempresa × periodo`, **sparse**. Dic-2025: 22 filas. Horizontal bars; acento en petróleo; label con nombre corto. No usar este ranking como headline.


In [ ]:
top_n = companies.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.barh(top_n["empresa"], top_n["prod_pet_m3"] / 1e3, color=BARRILITO["accent"])
bars[-1].set_color(BARRILITO["accent2"])  # primero = último en lista invertida
ax.set_title("Top 10 empresas · dic-2025 · petróleo (miles de m³)")
ax.set_xlabel("10³ m³")
ax.xaxis.set_major_formatter(thousands_axis())
ax.grid(True, axis="x")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

share_ypf = companies.iloc[0].prod_pet_m3 / companies.prod_pet_m3.sum()
print(f"YPF share dic-2025: {100 * share_ypf:.1f}%  ·  n={len(companies)}")


## 5. Vista de área (permiso / concesión)

Grano preferido: `idareapermisoconcesion`. No yacimiento. Top 10 + nota de 83 áreas en el mes.


In [ ]:
top_a = areas.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(top_a["areapermisoconcesion"], top_a["prod_pet_m3"] / 1e3, color=BARRILITO["line"])
ax.set_title("Top 10 áreas (permiso/concesión) · dic-2025 · petróleo (miles de m³)")
ax.set_xlabel("10³ m³")
ax.xaxis.set_major_formatter(thousands_axis())
ax.grid(True, axis="x")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
print("áreas en el mes:", len(areas), " · top:", areas.iloc[0].areapermisoconcesion)


## 6. Serie de las 5 empresas del último mes

Sparse: no todas están los 12 meses. Líneas, no stacked 100% (el total Pulse no es esas 5).


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = (
    top5[top5["periodo"] == top5["periodo"].max()]
    .sort_values("prod_pet_m3", ascending=False)["empresa"]
    .tolist()
)
palette = [BARRILITO["accent"], BARRILITO["accent2"], BARRILITO["line"], "#D9C3A0", "#8C7B6B"]
for i, name in enumerate(order):
    g = top5[top5["empresa"] == name].sort_values("periodo")
    ax.plot(g["periodo"], g["prod_pet_m3"] / 1e3, label=name, color=palette[i], linewidth=2.2)
ax.set_title("Top 5 de dic-2025 a lo largo de 2025 (miles de m³)")
ax.set_ylabel("10³ m³")
ax.legend(frameon=True, fontsize=8)
ax.grid(True, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 7. Completaciones — empty state (MUST)

No hay `fct_completions`. La UI **dice** que no hay dato. Cero curva inventada, cero “próximamente” con un gráfico vacío que parezca 0 fracturas reales.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.set_axis_off()
ax.set_facecolor(BARRILITO["surface"])
ax.text(0.04, 0.62, "Completaciones", fontsize=14, color=BARRILITO["text"], fontweight="bold", transform=ax.transAxes)
ax.text(
    0.04, 0.28,
    "Sin dato. Adjunto IV no está en el job Meltano default.\n"
    "Cuando exista fct_completions, esta tarjeta se reemplaza. No interpolar fracturas.",
    fontsize=11, color=BARRILITO["empty"], transform=ax.transAxes,
)
plt.tight_layout()
plt.show()


## 8. Mapa de componentes → Hito 3 (Tremor)

| Spike | Next / Tremor | Fuente |
| --- | --- | --- |
| Número `barriles_hoy` | `Metric` grande + `setInterval` 1s (server trae `rate_bbl_dia` + `periodo`) | `fct_barrilito_rate` |
| Disclaimer | texto MUST visible, `aria-live="polite"` | columna `disclaimer` |
| KPIs ritmo / m³ / well-months | `Card` / `Flex` | misma fila |
| Área chart petróleo mensual | `AreaChart` | `monthly_pulse` (o agg `fct_company_month`) |
| Barras empresas | `BarList` | `fct_company_month` último `periodo` |
| Barras áreas | `BarList` | `fct_area_month` último `periodo` |
| Líneas top 5 | `LineChart` | mismo mart, filter top ids |
| Empty fracturas | `Callout` color muted | no mart |

**Auth:** SA de Front con READ solo `marts_cap4_dev`. Nunca `GCP_SA_KEY` Meltano ni JSON en el bundle.

**Hosting:** sigue abierto (Vercel / Cloud Run). El spike no decide eso.

**Fuera:** mapa GIS, API pública, Barrilito-por-empresa como headline, recalcular bbl en el cliente.


## 9. Checklist de copy (aceptación de portada)

- [x] Contador interpolado desde `rate_bbl_dia` (reloj diario).
- [x] Disclaimer MUST pegado al número.
- [x] Ritmo + mes Cap. IV visibles.
- [x] Volumen del mes en m³, no en bbl inventados.
- [x] Ranking empresa y vista de área con grano documentado.
- [x] Completaciones = “sin dato”.
- [ ] Hito 3: pasar esto a Next. Este notebook no cierra el hito.
